# Best Model Selection and Hyperparameter Tuning - Rachel Wantuch
## Import dataset and ensure its loaded properly

In [16]:
import pandas as pd

In [17]:
df=pd.read_csv("Loan_Train.csv")
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


## Prepare the Data for Modeling
### 1. Drop the column "Loan_ID"

In [18]:
df_1=df.drop("Loan_ID", axis=1)

### 2. Drop any rows missing data

In [19]:
df_2=df_1.dropna()

### 3. Convert Categorical Columns to Dummy Variables

In [20]:
df_dummies=pd.get_dummies(df_2, drop_first=True, dtype=int)
df_dummies.head()

#This seelct everything except the class column since that is our target variable.
#target_column='Loan_Status'
#df_dummies=pd.get_dummies(df.drop(columns=[target_column]),drop_first=True)
#df_final=pd.concat([df_dummies,df[[target_column]]],axis=1)
#column_to_move=df_final.pop('class')
#df_final.insert(0,'class',column_to_move)

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Gender_Male,Married_Yes,Dependents_1,Dependents_2,Dependents_3+,Education_Not Graduate,Self_Employed_Yes,Property_Area_Semiurban,Property_Area_Urban,Loan_Status_Y
1,4583,1508.0,128.0,360.0,1.0,1,1,1,0,0,0,0,0,0,0
2,3000,0.0,66.0,360.0,1.0,1,1,0,0,0,0,1,0,1,1
3,2583,2358.0,120.0,360.0,1.0,1,1,0,0,0,1,0,0,1,1
4,6000,0.0,141.0,360.0,1.0,1,0,0,0,0,0,0,0,1,1
5,5417,4196.0,267.0,360.0,1.0,1,1,0,1,0,0,1,0,1,1


## Split the Data into Training and Test Set. Target Variable= Loan_Status

In [21]:
from sklearn.model_selection import train_test_split
x=df_dummies.drop(columns=['Loan_Status_Y'])
y=df_dummies['Loan_Status_Y']
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42)

## Create a pipeline with a min-max scaler and a KNN classifier

In [62]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

In [63]:
standardizer=MinMaxScaler()
knn=KNeighborsClassifier(n_neighbors=5,n_jobs=-1)
pipeline=Pipeline([("standardizer",standardizer),("knn",knn)])

## Fit a default KNN classifier to the data with this pipeline. Reports the model accuracy.

In [64]:
pipeline.fit(x_train,y_train)
accuracy=pipeline.score(x_test,y_test)
print(accuracy)

0.78125


## Create a search space with parameter values from 1 to 10.

In [65]:
search_space=[{"knn__n_neighbors": [1,2,3,4,5,6,7,8,9,10]}]

## Fit a grid search with your pipeline, search space, and 5-fold cross validation

In [66]:
classifier=GridSearchCV(pipeline,search_space,cv=5,verbose=0)
classifier.fit(x_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('standardizer', MinMaxScaler()),
                                       ('knn',
                                        KNeighborsClassifier(n_jobs=-1))]),
             param_grid=[{'knn__n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}])

## Find the accuracy of the grid search best model on the test set

In [67]:
best_model=classifier.best_estimator_
accuracy=best_model.score(x_test,y_test)
print(accuracy)

0.7916666666666666


## Repeat steps 6 and 7 with the same pipeline, but expand your search space to include logistic regression and random forest models with the hyperparameter values.

In [49]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np

In [59]:
search_space=[{"classifier":[LogisticRegression(max_iter=500, solver='liblinear')],
                "classifier__penalty":['l1','l2'],
                "classifier__C":np.logspace(0,4,10)},
                {"classifier":[RandomForestClassifier()],
                 "classifier__n_estimators":[10,100,1000],
                 "classifier__max_features":[1,2,3]}]
pipeline=Pipeline([("classifier",RandomForestClassifier())])
classifier=GridSearchCV(pipeline,search_space,cv=5,verbose=0)

## What are the best model and hyperparameters found in the grid search? Find the accuracy of this model on the test set.

In [60]:
best_model=classifier.fit(x_train,y_train)
print(best_model.best_estimator_)

Pipeline(steps=[('classifier',
                 LogisticRegression(C=np.float64(7.742636826811269),
                                    max_iter=500, penalty='l1',
                                    solver='liblinear'))])


In [61]:
#Evaluate the model on the test set
y_pred=classifier.predict(x_test)
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)

0.8229166666666666


## Summarize your results

The default KNN (assuming I did it right) was accurate at 78.12%. Thats pretty good. If given 5 values it would get 4 of them into the right cluster.  By using GridSearcSV we improved the accuracy to 79.17% which is due in part to the function trying multiple attempts with different hyperparameters. For instance it settled on 3 as the optimal number of neighbors it looks like.

It improved even more with logistic regression model. The accuracy was 82.29%. I think this can be attributed to that its trying 500 iterations for an output where it gets classified into two options.